> **Notebook-first lesson.** Run cells top-to-bottom. Environment-specific operations are written to be inspectable even when a service/device is unavailable.

## Mathematical Framework

Math companions for this lesson:

- [Math 03 · Probability & Bayes](../../math/03_probability_bayes.ipynb)
- [Math 04 · Statistics & Likelihood](../../math/04_statistics_likelihood.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 14 · Signal/Radar Detection Mathematics](../../math/14_signal_detection_math.ipynb)

For this topic, explicitly state the **probabilistic/statistical model, objective, invariances, threshold/decision rule, and what assumptions connect the math to deployment data**.

# Lesson 52: Time-series foundations

## Time changes the split
A random split often leaks future information into training.

Prefer chronological splitting when forecasting future behavior.

## Concepts
- trend
- seasonality
- autocorrelation
- stationarity
- lag features
- rolling statistics
- forecasting horizon

## Baselines
Always compare against:
- last value
- moving average
- seasonal naive baseline where appropriate

## Example lag features


In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)
import numpy as np
import pandas as pd
rng = np.random.default_rng(SEED)
df = pd.DataFrame({'value': np.sin(np.arange(100)/10) + rng.normal(0,.1,100)})


In [ ]:
import pandas as pd

df["lag_1"] = df["value"].shift(1)
df["lag_5"] = df["value"].shift(5)
df["rolling_mean_10"] = df["value"].shift(1).rolling(10).mean()



## Exercise
Generate a noisy sinusoidal signal with drift. Build lag features and compare a linear model against a naive baseline.


## Runnable activity
Run this experiment and change at least one data, threshold, deployment, or systems assumption.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
rng=np.random.default_rng(0)
t=np.arange(500); y=np.sin(t/20)+.002*t+rng.normal(0,.1,len(t))
df=pd.DataFrame({"y":y})
for lag in [1,2,5,10]: df[f"lag_{lag}"]=df.y.shift(lag)
df=df.dropna()
split=350
tr,te=df.iloc[:split],df.iloc[split:]
features=[c for c in df.columns if c.startswith("lag_")]
m=LinearRegression().fit(tr[features],tr.y)
pred=m.predict(te[features]); naive=te["lag_1"]
print("linear RMSE",mean_squared_error(te.y,pred)**.5)
print("persistence RMSE",mean_squared_error(te.y,naive)**.5)

## Engineering checkpoint
Record the metric/result, the assumption you changed, and what would make this experiment invalid in a real deployment.

### Forecast timestamp convention

To predict value at time t, every feature must depend only on times before t. The rolling mean above is shifted by one step; including value[t] would leak the current target.